In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2024_ITO_Delhi_CPCB_2024.xlsx")

In [3]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,344.0,319.0,227.0,83.0,121.0,230.0,73.0,113.0,62.0,97.0,318.0,290.0
1,2,336.0,147.0,127.0,62.0,110.0,129.0,63.0,150.0,66.0,122.0,278.0,268.0
2,3,336.0,136.0,116.0,81.0,191.0,114.0,76.0,90.0,79.0,112.0,367.0,221.0
3,4,386.0,247.0,114.0,88.0,246.0,185.0,69.0,88.0,62.0,189.0,356.0,136.0
4,5,272.0,136.0,108.0,83.0,270.0,207.0,63.0,68.0,168.0,134.0,335.0,130.0
5,6,254.0,93.0,106.0,116.0,204.0,112.0,56.0,89.0,121.0,NaN,330.0,191.0
6,7,279.0,121.0,197.0,106.0,257.0,155.0,51.0,64.0,88.0,NaN,358.0,250.0
7,8,290.0,111.0,126.0,114.0,160.0,184.0,52.0,64.0,81.0,NaN,361.0,316.0
8,9,324.0,162.0,112.0,173.0,155.0,119.0,60.0,64.0,85.0,192.0,341.0,199.0
9,10,197.0,302.0,162.0,207.0,159.0,111.0,96.0,83.0,104.0,115.0,332.0,257.0


In [4]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [5]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [6]:
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [7]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,344.000000,319.000000,227.000000,83.000000,121.0,118.970588,73.000000,73.896552,62.000000,97.000000,318.000000,290.0
1,2,336.000000,147.000000,127.000000,62.000000,110.0,129.000000,63.000000,73.896552,66.000000,122.000000,278.000000,268.0
2,3,336.000000,136.000000,116.000000,81.000000,191.0,114.000000,76.000000,90.000000,79.000000,112.000000,367.000000,221.0
3,4,386.000000,247.000000,114.000000,88.000000,246.0,118.970588,69.000000,88.000000,62.000000,189.000000,356.000000,136.0
4,5,272.000000,136.000000,108.000000,83.000000,270.0,118.970588,63.000000,68.000000,86.545455,134.000000,335.000000,130.0
5,6,254.000000,93.000000,106.000000,116.000000,204.0,112.000000,56.000000,89.000000,121.000000,182.617647,330.000000,191.0
6,7,279.000000,121.000000,197.000000,106.000000,257.0,155.000000,51.000000,64.000000,88.000000,182.617647,358.000000,250.0
7,8,290.000000,111.000000,126.000000,114.000000,160.0,118.970588,52.000000,64.000000,81.000000,182.617647,361.000000,316.0
8,9,324.000000,162.000000,112.000000,173.000000,155.0,119.000000,60.000000,64.000000,85.000000,192.000000,341.000000,199.0
9,10,197.000000,302.000000,162.000000,207.000000,159.0,111.000000,96.000000,83.000000,104.000000,115.000000,332.000000,257.0
